In [ ]:
from collections.abc import Callable

import equinox as eqx
import jax
import jax.numpy as jnp
from context_flux_no.models.dpot import TimeAggregator
from context_flux_no.nn.embedding import PatchEmbedding
from context_flux_no.nn.misc import to_ntuple
from context_flux_no.nn.operators import AdaptiveFourier
from context_flux_no.nn.position_encoding import (
    LearnedPositionEncoding,
)
from jaxtyping import Array, Float, PRNGKeyArray

(32, 100)

(32, 100, 100)

(32, 100, 100, 100)

In [ ]:
def get_grid_1d(x: Float[Array, "... dim"]) -> Float[Array, "... 1"]:
    *rest, n_x, _ = x.shape
    grid_x = jnp.expand_dims(jnp.linspace(0, 1, n_x), axis=-1)
    return jnp.broadcast_to(grid_x, shape=(*rest, n_x, 1))


class DPOTBlock1D(eqx.Module):
    norm1: eqx.nn.GroupNorm
    norm2: eqx.nn.GroupNorm
    spatial_mixing: AdaptiveFourier
    channel_mixing: eqx.nn.Sequential

    double_skip: bool = eqx.field(static=True)

    def __init__(
        self,
        channels: int,
        channels_hidden: int,
        groups: int = 8,
        double_skip: bool = True,
        activation: Callable = jax.nn.gelu,
        dtype=None,
        *,
        key: PRNGKeyArray,
    ):
        self.norm1 = eqx.nn.GroupNorm(groups, channels, dtype=dtype)
        self.norm2 = eqx.nn.GroupNorm(groups, channels, dtype=dtype)

        keys = jax.random.split(key, 3)
        self.spatial_mixing = AFNO1D()
        self.channel_mixing = eqx.nn.Sequential(
            [
                eqx.nn.Conv(
                    num_spatial_dims=1,
                    in_channels=channels,
                    out_channels=channels_hidden,
                    kernel_size=1,
                    stride=1,
                    dtype=dtype,
                    key=keys[1],
                ),
                eqx.nn.Lambda(jax.nn.gelu),
                eqx.nn.Conv(
                    num_spatial_dims=1,
                    in_channels=channels_hidden,
                    out_channels=channels,
                    kernel_size=1,
                    stride=1,
                    dtype=dtype,
                    key=keys[2],
                ),
            ]
        )
        self.double_skip = double_skip

    def __call__(
        self,
        x: Float[Array, "channels patch_x"],
    ) -> Float[Array, "channels patch_x"]:
        y = self.spatial_mixing(self.norm1(x))

        if self.double_skip:
            y = y + x
            x = y

        y = self.channel_mixing(self.norm2(y))
        y = y + x
        return y


class DPOT1D(eqx.Module):
    patch_embedding: PatchEmbedding
    position_embedding: LearnedPositionEncoding
    time_aggregator: TimeAggregator
    blocks: list[DPOTBlock1D]
    output_head: eqx.nn.Sequential
    cls_head: eqx.nn.Sequential

    def __init__(
        self,
        in_channels: int,
        in_timesteps: int,
        patch_size: int | tuple[int],
        embedding_dim: int,
        max_frequency_modes: int | tuple[int],
        fno_depth: int,
        hidden_dim_patch: int,
        activation: Callable[[Array], Array] = jax.nn.gelu,
        dtype=None,
        *,
        key: PRNGKeyArray,
    ):
        keys = jax.random.split(key, 5)

        patch_size = to_ntuple(patch_size, 1)
        self.patch_embedding = PatchEmbedding(
            num_spatial_dims=1,
            patch_size=patch_size,
            in_dim=in_channels + 1,
            embedding_dim=embedding_dim,
            num_hidden=1,
            hidden_dim=hidden_dim_patch,
            activation=activation,
            dtype=dtype,
            key=keys[0],
        )
        self.position_embedding = LearnedPositionEncoding(
            spatial_dims=patch_size, channels=embedding_dim
        )
        self.time_aggregator = TimeAggregator(
            timesteps=in_timesteps, channels=embedding_dim, dtype=dtype, key=keys[1]
        )
        self.blocks = [DPOTBlock1D()]

    def __call__(
        self, u: Float[Array, "time channels grid_x"]
    ) -> Float[Array, "channels grid_x"]:
        # TODO: Check if normalization is actually used and implement if necessary
        grid = get_grid_1d(u)
        u: Float[Array, "time channels+1 grid_x"] = jnp.concatenate((u, grid), axis=1)

        # Patch embedding for the spatial dimensions
        v: Float[Array, "time channels_embed patch_x"] = eqx.filter_vmap(
            self.patch_embedding
        )(u)

        # Apply positional embedding
        v: Float[Array, "time channels_embed patch_x"] = eqx.filter_vmap(
            self.position_embedding
        )(v)

        # Time aggregation layer
        v: Float[Array, "channels_embed patch_x"] = self.time_aggregator(v)

        for dpot_block in self.blocks:
            v: Float[Array, "channels_embed patch_x"] = dpot_block(v)

        u_next = self.output_head(v)

        cls_token = jnp.mean(v, axis=-1)
        cls_pred = self.cls_head(cls_token)
        return u_next, cls_pred